In [ ]:
import pandas as pd
df = pd.read_csv("../data/cleaned_recipes.csv")
df.head()

In [ ]:
def parse_ingredients(ingredient_string):
    ingredients = [
        i.strip().lower()
        for i in ingredient_string.split(",")
        if i.strip()
    ]
    # Known bug: can damage real ingredients beginning with c.
    if ingredients and ingredients[0].startswith("c"):
        ingredients[0] = ingredients[0][1:].strip()
    return ingredients

df["ingredient_list"] = df["cleaned_ingredients"].apply(parse_ingredients)

In [ ]:
df[["Name", "ingredient_list"]].head(5)

In [ ]:
user_ingredients = set(["blueberries", "sugar", "vanilla yogurt", "flour", "eggs"])

In [ ]:
def ingredient_match_score(recipe_ingredients, user_ingredients):
    matches = set(recipe_ingredients) & set(user_ingredients)
    if not recipe_ingredients:
        return 0
    return len(matches) / len(recipe_ingredients)

In [ ]:
df["match_score"] = df["ingredient_list"].apply(
    lambda x: ingredient_match_score(x, user_ingredients)
)
df[["Name", "match_score"]].sort_values(
    by="match_score", ascending=False
).head(10)

In [ ]:
filtered = df[df["match_score"] >= 0.3]
len(filtered)

In [ ]:
def only_user_ingredients(row, user_ingredients):
    recipe_set = set([i.strip().lower() for i in row["ingredient_list"]])
    return recipe_set.issubset(user_ingredients)

filtered_strict = filtered[filtered.apply(lambda row: only_user_ingredients(row, user_ingredients), axis=1)].copy()

filtered = df[df["match_score"] >= 0.3].copy()
max_reviews = filtered["ReviewCount"].max()
filtered["review_weight"] = filtered["ReviewCount"] / max_reviews
filtered["weighted_rating"] = (
    (filtered["AggregatedRating"].fillna(0) * filtered["review_weight"]) +
    (2.5 * (1 - filtered["review_weight"]))
)
filtered["final_score"] = (
    filtered["match_score"] * 0.7 +
    (filtered["weighted_rating"] / 5) * 0.3
)
filtered_strict = filtered[filtered.apply(lambda row: set(row["ingredient_list"]).issubset(user_ingredients), axis=1)].copy()

In [ ]:
filtered.sort_values(
    by="final_score",
    ascending=False
)[["RecipeId", "Name", "AuthorName", "match_score", "AggregatedRating", "ReviewCount", "final_score"]].head(25)

In [ ]:
filtered_strict.sort_values(
    by="final_score",
    ascending=False
)[["RecipeId", "Name", "AuthorName", "match_score", "AggregatedRating", "ReviewCount", "final_score"]].head(25)